<a href="https://colab.research.google.com/github/sabrinayusupova/2026_Fall_Application-Deployment/blob/main/20261023/chinook_db_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chinook DB 실습 노트북 요약 (Google Colab 기반)

## 1. 데이터 준비
- `requests`로 Chinook SQLite DB 다운로드 (GitHub 원본)
- `os.path.exists`로 파일 존재 여부를 확인하여 중복 다운로드 방지 (함수화)

## 2. SQLite 기본 사용법
- `sqlite3.connect()`로 DB 연결, `cursor.execute()`로 쿼리 실행
- `sqlite_master` 조회로 전체 테이블 목록 확인
- `with` 구문으로 연결 관리 (자동 종료·커밋)
- 연결 종료 후 재사용 시 발생하는 오류 확인 (연결 재설정 필요)

## 3. pandas 연동
- `pd.read_sql_query()`로 테이블을 DataFrame으로 로드
- 전체 테이블을 반복문으로 읽어 딕셔너리(`dfs`)에 저장

## 4. 데이터 프로파일링 및 엑셀 출력
- `summarize_df()` 사용자 정의 함수 작성
  - 컬럼별 데이터 타입, 고유값 수, 결측치 수·비율, 대표값 샘플 요약
- `xlsxwriter` + `pd.ExcelWriter`로 테이블별 시트 분리 저장
- `google.colab.files.download()`로 결과 파일 다운로드

## 5. Supabase 연동
- Colab `userdata`로 API 키를 안전하게 관리
- `socket.gethostbyname`으로 DNS 사전 점검
- `supabase` 파이썬 클라이언트 생성

## 6. SQLite → Supabase 데이터 이관 (ETL)
- 테이블·컬럼 매핑 정의 (PascalCase → snake_case 변환)
- 외래키 의존성 순서대로 적재 (Artist → Album → Track → Employee → Customer → Invoice …)
- `BATCH_SIZE=500` 단위 배치 업로드 (API 페이로드 제한 대응)
- `try/except`로 오류 처리

## 7. 트러블슈팅 (NaN 오류 해결)
- **원인**: `employee.reports_to` 등 정수형 NULL이 pandas에서 `NaN(float)`로 변환되어 JSON 직렬화 오류 발생
- **연쇄 영향**: employee 실패 → customer → invoice → invoice_line까지 외래키 오류로 전부 실패
- **해결**: `replace({np.nan: None})` 적용, `pd.isna()` 재검사, 정수형 float을 `int`로 변환
- 기존 테이블 삭제 후 재이관

## 핵심 학습 포인트
- 로컬 DB(SQLite) → 클라우드 DB(Supabase) 마이그레이션 전 과정
- 결측치(NaN/None)의 JSON 변환 문제 및 외래키 순서 관리의 중요성

In [1]:
DB_URL = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
DB_PATH = "chinook.db"

In [2]:
os.path.exists(DB_PATH)

NameError: name 'os' is not defined

In [ ]:
import os
import requests

def download_chinook_db():
    if not os.path.exists(DB_PATH):
        print("📥 Downloading Chinook database...")
        response = requests.get(DB_URL)
        with open(DB_PATH, "wb") as f:
            f.write(response.content)
        print("✅ Download complete.")
    else:
        print("✅ Chinook DB already exists.")

download_chinook_db()

In [ ]:
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect(DB_PATH)

# Get a list of all tables in the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables in the database:", [table[0] for table in tables])
conn.close()

In [ ]:
import pandas as pd
import sqlite3

# Read data from a table into a pandas DataFrame
# Replace 'albums' with one of the available table names listed above.
query = "SELECT * FROM Album"
df = pd.read_sql_query(query, conn)

# Close the connection
conn.close()

# Display the first few rows of the DataFrame
df

In [ ]:
# Re-establish connection to the SQLite database
conn = sqlite3.connect(DB_PATH)

# Read data from a table into a pandas DataFrame
# Replace 'albums' with one of the available table names listed above.
query = "SELECT * FROM Album"
df = pd.read_sql_query(query, conn)

# Close the connection
conn.close()

# Display the first few rows of the DataFrame
df

In [ ]:
# SQLite 데이터베이스 연결 및 pandas로 Album 테이블 읽기
with sqlite3.connect(DB_PATH) as conn:
    query = "SELECT * FROM Album"
    df_album = pd.read_sql_query(query, conn)

# 결과 확인
df_album.head()

In [ ]:
# @title sqlite3.connect()를 with 구문에 직접 포함
import sqlite3

# Connect to the SQLite database
with sqlite3.connect('chinook.db') as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print("Available tables in the database:", [table[0] for table in tables])

In [ ]:
# @title 모든 테이블을 사전형으로 읽기
# Initialize a dictionary to store all DataFrames
dfs = {}

# Connect to the SQLite database using a 'with' statement
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    # Iterate through each table and load it into a DataFrame
    for table_name in [table[0] for table in tables]:
        query = f"SELECT * FROM {table_name}"
        df = pd.read_sql_query(query, conn)
        dfs[table_name] = df
        print(f"--- {table_name} DataFrame head ---")
        display(df.head())
        print("\n")

print("All tables loaded into 'dfs' dictionary.")

In [ ]:
dfs.keys()

In [ ]:
dfs['Album']

In [ ]:
def summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    # 데이터프레임의 각 컬럼별 데이터 타입, 결측치 수/비율, 대표값 5개를 요약한 DataFrame을 반환합니다.
    summary_data = []
    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        null_cnt = df[col].isnull().sum()
        null_ratio = round((null_cnt / len(df)) * 100, 2)
        unique_vals = df[col].dropna().unique()
        sample_vals = list(unique_vals[:7])

        summary_data.append({
            'Column': col,
            'N_Unique': n_unique,
            'Dtype': dtype,
            'Null_Count': null_cnt,
            'Null_Ratio(%)': null_ratio,
            'Sample_Values_5': sample_vals
        })

    return pd.DataFrame(summary_data)

In [ ]:
# Install the xlsxwriter library
!pip install xlsxwriter

In [ ]:
output_excel_path = 'all_tables_summary.xlsx'

# Create an Excel writer object
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    for table_name, df in dfs.items():
        print(f"Summarizing {table_name}...")
        summary_df = summarize_df(df)
        # Write each summary DataFrame to a separate sheet
        summary_df.to_excel(writer, sheet_name=table_name, index=False)
        print(f"Summary for {table_name} written to sheet '{table_name}'.")

print(f"All table summaries saved to '{output_excel_path}'")

In [ ]:
from google.colab import files

# Download the generated Excel file
files.download('all_tables_summary.xlsx')

In [ ]:
# @title supabase
from google.colab import userdata
SUPABASE_KEY = userdata.get('supakey')

# 접속이 되는 지 사전 확인
import socket
from urllib.parse import urlparse

SUPABASE_URL = "https://qctwuimxnppnzmjcnluc.supabase.co"
hostname = urlparse(SUPABASE_URL).hostname

try:
    ip = socket.gethostbyname(hostname)
    print(f"DNS 조회 성공: {ip}")
except socket.gaierror as e:
    print(f"DNS 조회 실패: {e}")

In [ ]:
!pip install -qU supabase

from supabase.client import create_client, Client

# 수파베이스 공식 파이썬 클라이언트 객체를 생성합니다.
supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
import sqlite3
import pandas as pd

# 1. SQLite 데이터베이스 연결
conn = sqlite3.connect(DB_PATH)

# 2. 이관할 테이블 목록 및 컬럼 매핑 (외래키 의존성 순서대로 정렬)
# (SQLite 카멜케이스/파스칼케이스 컬럼명을 Supabase의 스네이크케이스 컬럼명으로 전환)
table_mappings = [
    {
        "sqlite_table": "Artist",
        "supabase_table": "artist",
        "columns": {"ArtistId": "artist_id", "Name": "name"}
    },
    {
        "sqlite_table": "Genre",
        "supabase_table": "genre",
        "columns": {"GenreId": "genre_id", "Name": "name"}
    },
    {
        "sqlite_table": "MediaType",
        "supabase_table": "media_type",
        "columns": {"MediaTypeId": "media_type_id", "Name": "name"}
    },
    {
        "sqlite_table": "Album",
        "supabase_table": "album",
        "columns": {"AlbumId": "album_id", "Title": "title", "ArtistId": "artist_id"}
    },
    {
        "sqlite_table": "Track",
        "supabase_table": "track",
        "columns": {
            "TrackId": "track_id", "Name": "name", "AlbumId": "album_id",
            "MediaTypeId": "media_type_id", "GenreId": "genre_id",
            "Composer": "composer", "Milliseconds": "milliseconds",
            "Bytes": "bytes", "UnitPrice": "unit_price"
        }
    },
    {
        "sqlite_table": "Employee",
        "supabase_table": "employee",
        "columns": {
            "EmployeeId": "employee_id", "LastName": "last_name", "FirstName": "first_name",
            "Title": "title", "ReportsTo": "reports_to", "BirthDate": "birth_date",
            "HireDate": "hire_date", "Address": "address", "City": "city",
            "State": "state", "Country": "country", "PostalCode": "postal_code",
            "Phone": "phone", "Fax": "fax", "Email": "email"
        }
    },
    {
        "sqlite_table": "Customer",
        "supabase_table": "customer",
        "columns": {
            "CustomerId": "customer_id", "FirstName": "first_name", "LastName": "last_name",
            "Company": "company", "Address": "address", "City": "city",
            "State": "state", "Country": "country", "PostalCode": "postal_code",
            "Phone": "phone", "Fax": "fax", "Email": "email", "SupportRepId": "support_rep_id"
        }
    },
    {
        "sqlite_table": "Invoice",
        "supabase_table": "invoice",
        "columns": {
            "InvoiceId": "invoice_id", "CustomerId": "customer_id", "InvoiceDate": "invoice_date",
            "BillingAddress": "billing_address", "BillingCity": "billing_city",
            "BillingState": "billing_state", "BillingCountry": "billing_country",
            "BillingPostalCode": "billing_postal_code", "Total": "total"
        }
    },
    {
        "sqlite_table": "InvoiceLine",
        "supabase_table": "invoice_line",
        "columns": {
            "InvoiceLineId": "invoice_line_id", "InvoiceId": "invoice_id",
            "TrackId": "track_id", "UnitPrice": "unit_price", "Quantity": "quantity"
        }
    },
    {
        "sqlite_table": "Playlist",
        "supabase_table": "playlist",
        "columns": {"PlaylistId": "playlist_id", "Name": "name"}
    },
    {
        "sqlite_table": "PlaylistTrack",
        "supabase_table": "playlist_track",
        "columns": {"PlaylistId": "playlist_id", "TrackId": "track_id"}
    }
]

# 3. 데이터 이관 진행
print("🚀 Supabase 데이터 이관을 시작합니다...\n")

BATCH_SIZE = 500  # 청크 단위 업로드 (API 페이로드 제한 방지)

for mapping in table_mappings:
    s_table = mapping["sqlite_table"]
    target_table = mapping["supabase_table"]
    col_map = mapping["columns"]

    # SQLite 데이터 읽기
    df = pd.read_sql_query(f'SELECT * FROM "{s_table}"', conn)

    # 컬럼명 변경 및 NaN(결측치) 처리
    df = df.rename(columns=col_map)
    df = df.where(pd.notnull(df), None)

    records = df.to_dict(orient='records')
    total_records = len(records)

    print(f"📦 [{target_table}] 테이블 이관 중 ({total_records} 개 레코드)...")

    # 배치 분할 업로드
    for i in range(0, total_records, BATCH_SIZE):
        batch = records[i:i + BATCH_SIZE]
        try:
            supabase_client.table(target_table).insert(batch).execute()
        except Exception as e:
            print(f"   ⚠️ 오류 발생 ({target_table}): {e}")
            break

    print(f"   ✅ [{target_table}] 완료\n")

conn.close()
print("🎉 모든 Chinook 데이터베이스 이관이 완료되었습니다!")

- employee 테이블을 넣을 때 발생한 NaN (float) 오류 때문

- employee 테이블의 reports_to 컬럼 등 숫자로 되어 있는 빈 값(Null)이 Pandas에서 NaN (float)으로 처리되어 Supabase JSON 변환 중 에러가 발생

> 이로 인해 employee 데이터가 입력되지 않았고, 이후 customer → invoice → invoice_line 테이블들이 외래키(Foreign Key) 참조 연쇄 오류로 전부 실패한 것

- 해결 방법
Pandas의 where(pd.notnull(df), None) 대신 replace({np.nan: None}) 및 None 변환 로직을 보완하면 깔끔하게 해결

### 모든 테이블을 삭제하고 SQL문장 다시 수행하기

In [3]:
import sqlite3
import pandas as pd
import numpy as np

# 1. SQLite 데이터베이스 연결
conn = sqlite3.connect(DB_PATH)

# 2. 이관할 테이블 목록 및 컬럼 매핑
table_mappings = [
    {
        "sqlite_table": "Artist",
        "supabase_table": "artist",
        "columns": {"ArtistId": "artist_id", "Name": "name"}
    },
    {
        "sqlite_table": "Genre",
        "supabase_table": "genre",
        "columns": {"GenreId": "genre_id", "Name": "name"}
    },
    {
        "sqlite_table": "MediaType",
        "supabase_table": "media_type",
        "columns": {"MediaTypeId": "media_type_id", "Name": "name"}
    },
    {
        "sqlite_table": "Album",
        "supabase_table": "album",
        "columns": {"AlbumId": "album_id", "Title": "title", "ArtistId": "artist_id"}
    },
    {
        "sqlite_table": "Track",
        "supabase_table": "track",
        "columns": {
            "TrackId": "track_id", "Name": "name", "AlbumId": "album_id",
            "MediaTypeId": "media_type_id", "GenreId": "genre_id",
            "Composer": "composer", "Milliseconds": "milliseconds",
            "Bytes": "bytes", "UnitPrice": "unit_price"
        }
    },
    {
        "sqlite_table": "Employee",
        "supabase_table": "employee",
        "columns": {
            "EmployeeId": "employee_id", "LastName": "last_name", "FirstName": "first_name",
            "Title": "title", "ReportsTo": "reports_to", "BirthDate": "birth_date",
            "HireDate": "hire_date", "Address": "address", "City": "city",
            "State": "state", "Country": "country", "PostalCode": "postal_code",
            "Phone": "phone", "Fax": "fax", "Email": "email"
        }
    },
    {
        "sqlite_table": "Customer",
        "supabase_table": "customer",
        "columns": {
            "CustomerId": "customer_id", "FirstName": "first_name", "LastName": "last_name",
            "Company": "company", "Address": "address", "City": "city",
            "State": "state", "Country": "country", "PostalCode": "postal_code",
            "Phone": "phone", "Fax": "fax", "Email": "email", "SupportRepId": "support_rep_id"
        }
    },
    {
        "sqlite_table": "Invoice",
        "supabase_table": "invoice",
        "columns": {
            "InvoiceId": "invoice_id", "CustomerId": "customer_id", "InvoiceDate": "invoice_date",
            "BillingAddress": "billing_address", "BillingCity": "billing_city",
            "BillingState": "billing_state", "BillingCountry": "billing_country",
            "BillingPostalCode": "billing_postal_code", "Total": "total"
        }
    },
    {
        "sqlite_table": "InvoiceLine",
        "supabase_table": "invoice_line",
        "columns": {
            "InvoiceLineId": "invoice_line_id", "InvoiceId": "invoice_id",
            "TrackId": "track_id", "UnitPrice": "unit_price", "Quantity": "quantity"
        }
    },
    {
        "sqlite_table": "Playlist",
        "supabase_table": "playlist",
        "columns": {"PlaylistId": "playlist_id", "Name": "name"}
    },
    {
        "sqlite_table": "PlaylistTrack",
        "supabase_table": "playlist_track",
        "columns": {"PlaylistId": "playlist_id", "TrackId": "track_id"}
    }
]

# 3. 데이터 이관 진행
print("🚀 Supabase 데이터 이관을 시작합니다...\n")

BATCH_SIZE = 500

for mapping in table_mappings:
    s_table = mapping["sqlite_table"]
    target_table = mapping["supabase_table"]
    col_map = mapping["columns"]

    # SQLite 데이터 읽기
    df = pd.read_sql_query(f'SELECT * FROM "{s_table}"', conn)

    # 컬럼명 변경
    df = df.rename(columns=col_map)

    # NaN/NaT 값을 Python None(JSON null)으로 완벽하게 변환하는 부분
    df = df.replace({np.nan: None})
    records = df.to_dict(orient='records')

    # dict 요소 중 float NaN 또는 NaT를 한번 더 체크하여 None 처리
    clean_records = []
    for record in records:
        clean_record = {
            k: (None if pd.isna(v) else int(v) if isinstance(v, float) and v.is_integer() else v)
            for k, v in record.items()
        }
        clean_records.append(clean_record)

    total_records = len(clean_records)
    print(f"📦 [{target_table}] 테이블 이관 중 ({total_records} 개 레코드)...")

    # 배치 분할 업로드
    success = True
    for i in range(0, total_records, BATCH_SIZE):
        batch = clean_records[i:i + BATCH_SIZE]
        try:
            supabase_client.table(target_table).insert(batch).execute()
        except Exception as e:
            print(f"   ⚠️ 오류 발생 ({target_table}): {e}")
            success = False
            break

    if success:
        print(f"   ✅ [{target_table}] 성공\n")

conn.close()
print("🎉 모든 Chinook 데이터베이스 이관이 완벽히 완료되었습니다!")

🚀 Supabase 데이터 이관을 시작합니다...

📦 [artist] 테이블 이관 중 (275 개 레코드)...
   ⚠️ 오류 발생 (artist): name 'supabase_client' is not defined
📦 [genre] 테이블 이관 중 (25 개 레코드)...
   ⚠️ 오류 발생 (genre): name 'supabase_client' is not defined
📦 [media_type] 테이블 이관 중 (5 개 레코드)...
   ⚠️ 오류 발생 (media_type): name 'supabase_client' is not defined
📦 [album] 테이블 이관 중 (347 개 레코드)...
   ⚠️ 오류 발생 (album): name 'supabase_client' is not defined
📦 [track] 테이블 이관 중 (3503 개 레코드)...
   ⚠️ 오류 발생 (track): name 'supabase_client' is not defined
📦 [employee] 테이블 이관 중 (8 개 레코드)...
   ⚠️ 오류 발생 (employee): name 'supabase_client' is not defined
📦 [customer] 테이블 이관 중 (59 개 레코드)...
   ⚠️ 오류 발생 (customer): name 'supabase_client' is not defined
📦 [invoice] 테이블 이관 중 (412 개 레코드)...
   ⚠️ 오류 발생 (invoice): name 'supabase_client' is not defined
📦 [invoice_line] 테이블 이관 중 (2240 개 레코드)...
   ⚠️ 오류 발생 (invoice_line): name 'supabase_client' is not defined
📦 [playlist] 테이블 이관 중 (18 개 레코드)...
   ⚠️ 오류 발생 (playlist): name 'supabase_client' is not define